In [4]:
import pandas as pd
from pathlib import Path
Dataset = Path("Dataset")
products = pd.read_csv(Dataset/'olist_products_dataset.csv')
translation = pd.read_csv(Dataset/'product_category_name_translation.csv')

In [5]:
products = pd.merge(products, translation, on='product_category_name', how='left')

In [6]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [7]:
products.drop(columns=['product_category_name'], inplace=True)

In [8]:
products.rename(columns={"product_category_name_english":"product_category"},inplace=True)

In [9]:
geo = pd.read_csv(Dataset/'olist_geolocation_dataset.csv')

In [10]:
geo_clean = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'median',
    'geolocation_lng': 'median',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

In [11]:
payments = pd.read_csv(Dataset/'olist_order_payments_dataset.csv')

In [12]:
payments_clean = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max'
}).reset_index()

In [14]:
sellers = pd.read_csv(Dataset/'olist_sellers_dataset.csv')

In [15]:
sellers['seller_city'] = sellers['seller_city'].str.lower().str.strip()

In [16]:
city_map = {
    'sao paulo': 'sao paulo',
    'são paulo': 'sao paulo',
    'saopaulo': 'sao paulo',
    'bragança paulista': 'braganca paulista',
    'belo horizonte': 'belo horizonte',
    'rio de janeiro': 'rio de janeiro'
}

In [17]:
sellers['seller_city'] = sellers['seller_city'].replace(city_map)

In [18]:
orders = pd.read_csv(Dataset/'olist_orders_dataset.csv')

In [19]:
date_cols = ['order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [20]:

orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

In [21]:
orders['is_late'] = orders['delivery_delay_days'] > 0

In [23]:
df = pd.merge(orders, payments_clean, on='order_id', how='left')

In [24]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,payment_value,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,False,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,False,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,False,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,False,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,False,28.62,1.0
